# Data: BRFSS Survey Data in several years [Preprocess]

Links: https://www.cdc.gov/brfss/annual_data/annual_*.html (*: 2024, 2023, 2022)

---

In [1]:
import glob
import os
import re

import pandas as pd
import numpy as np

# Load

In [2]:
# Constants

n_samples = 40000 # Number of samples to limit dataset (static) ~ 75 MB

In [3]:
# ============================================================
# Run this cell just one time
# Dynamic Loading of Multiple Years of BRFSS Data
# Structure: data/2024/LLCP2024.XPT, data/2023/LLCP2023.XPT, ...
# ============================================================

n_years = 0       # Number of total datasets (dynamic)

# Find all .XPT files directly under year subdirectories
# Pattern: data/*/LLCP*.XPT
xpt_files = glob.glob("data/*/LLCP*.XPT")

print(f"Found {len(xpt_files)} XPT files:")
for f in sorted(xpt_files):
    print(f"  {f}")

if not xpt_files:
    raise FileNotFoundError("No .XPT files found in data/*/LLCP*.XPT.")

# Read and merge all files
dfs = []
for file_path in sorted(xpt_files):  # sort to ensure consistent order
    # Extract year from folder name (e.g., data/2024/LLCP2024.XPT -> 2024)
    folder = os.path.basename(os.path.dirname(file_path))  # e.g., "2024"
    year_match = re.search(r'(\d{4})', folder)
    if year_match:
        year = int(year_match.group(1))
    else:
        # Fallback: try filename
        filename = os.path.basename(file_path)
        year_match = re.search(r'(\d{4})', filename)
        if year_match:
            year = int(year_match.group(1))
        else:
            year = None
            print(f"Warning: Could not extract year from {file_path}")

    print(f"Loading {file_path} (year {year})...")
    df_year = pd.read_sas(file_path, format='xport')
    
    if year is not None:
        df_year['YEAR'] = year
    dfs.append(df_year)
    n_years += 1

Found 3 XPT files:
  data\2022\LLCP2022.XPT
  data\2023\LLCP2023.XPT
  data\2024\LLCP2024.XPT
Loading data\2022\LLCP2022.XPT (year 2022)...
Loading data\2023\LLCP2023.XPT (year 2023)...
Loading data\2024\LLCP2024.XPT (year 2024)...


In [4]:
# Concatenate all years
df = pd.concat(dfs, ignore_index=True)

print(f"Total rows: {len(df)}, columns: {len(df.columns)}")

# ============================================================
# Ensure consistent columns across years
# ============================================================
# Get intersection of columns across all dataframes
common_cols = None
for df_year in dfs:
    cols = set(df_year.columns)
    if common_cols is None:
        common_cols = cols
    else:
        common_cols = common_cols.intersection(cols)

common_cols = list(common_cols)

# Remove 'YEAR' from common columns if it was added (it's present in all years)
if 'YEAR' in common_cols:
    common_cols.remove('YEAR')

print(f"Number of common columns across all years: {len(common_cols)}")

# Keep only common columns plus YEAR (if present)
year_col = 'YEAR' if 'YEAR' in df.columns else None
cols_to_keep = common_cols + ([year_col] if year_col else [])
df = df[cols_to_keep]

print(f"Final merged dataset shape: {df.shape}")
print(f"Years included: {sorted(df['YEAR'].unique()) if year_col else 'N/A'}")

Total rows: 1336125, columns: 478
Number of common columns across all years: 224
Final merged dataset shape: (1336125, 225)
Years included: [np.int64(2022), np.int64(2023), np.int64(2024)]


In [5]:
feature_names = list(df.columns)
print(feature_names)

print(df.shape)

m, n = df.shape

print(f"m: {m}")
print(f"n: {n}")


#    BRFSS uses special codes instead of leaving fields blank:
#    7/77/777/7777 = Don't know / Not sure
#    8/88/888/8888 = Refused
#    9/99/999/9999 = Missing / Unknown
# ============================================================
# Standard missing codes (will apply to all OTHER columns)
standard_missing_codes = [
    7, 8, 9,
    77, 88, 99,
    777, 888, 999,
    7777, 8888, 9999
]

# For special variables, only remove extended missing codes
# (keep 7, 8, 9, and 88 as valid)
special_missing_codes = [
    77, 99,
    777, 999,
    7777, 9999
]

# Group 1: Health days (88 = 0 days, 7/8/9 = valid days)
health_days_vars = ['PHYSHLTH', 'MENTHLTH', 'POORHLTH']

# Group 2: Counts (7/8/9 = valid counts)
count_vars = [
    'NUMADULT', 'HHADULT', 'NUMHHOL4', 'NUMPHON4', 
    'CHILDREN', '_CHLDCNT'
]

# Group 3: Age values (7/8/9 = valid ages)
age_vars = ['DIABAGE4', 'CNCRAGE', '_AGE80']

# Group 4: Drinks (7/8/9 = valid number of drinks)
drink_vars = ['AVEDRNK4', 'MAXDRNKS', 'DRNK3GE5']

# Group 5: Cigarettes (7/8/9 = valid counts)
smoke_vars = ['LCSNUMCG', 'LCSNUMC_', '_PACKDAY']

# Group 6: Income levels (7/8/9 = valid brackets)
income_vars = ['INCOME3', '_INCOMG1']

# Group 7: Education levels (7/8/9 = valid levels)
education_vars = ['EDUCA', '_EDUCAG']

# Group 8: Time/frequency variables (7/8/9 = valid years)
time_vars = [
    'CHECKUP1', 'LASTDEN4', 'HOWLONG', 'LASTSIG4', 
    'LASTSMK2', 'COLNTES1', 'SIGMTES1', 'STOLTEST', 
    'VCLNTES2'
]

# Combine all special variables
special_vars = (
    health_days_vars + count_vars + age_vars + drink_vars + 
    smoke_vars + income_vars + education_vars + time_vars
)


# ============================================================
# FIPS Code Mapping Dictionaries
# ============================================================

# Dictionary 1: FIPS Code -> State Name
fips_to_state = {
    1: "Alabama", 2: "Alaska", 4: "Arizona", 5: "Arkansas",
    6: "California", 8: "Colorado", 9: "Connecticut", 10: "Delaware",
    11: "District of Columbia", 12: "Florida", 13: "Georgia", 15: "Hawaii",
    16: "Idaho", 17: "Illinois", 18: "Indiana", 19: "Iowa",
    20: "Kansas", 21: "Kentucky", 22: "Louisiana", 23: "Maine",
    24: "Maryland", 25: "Massachusetts", 26: "Michigan", 27: "Minnesota",
    28: "Mississippi", 29: "Missouri", 30: "Montana", 31: "Nebraska",
    32: "Nevada", 33: "New Hampshire", 34: "New Jersey", 35: "New Mexico",
    36: "New York", 37: "North Carolina", 38: "North Dakota", 39: "Ohio",
    40: "Oklahoma", 41: "Oregon", 42: "Pennsylvania", 44: "Rhode Island",
    45: "South Carolina", 46: "South Dakota", 47: "Tennessee", 48: "Texas",
    49: "Utah", 50: "Vermont", 51: "Virginia", 53: "Washington",
    54: "West Virginia", 55: "Wisconsin", 56: "Wyoming",
    66: "Guam", 72: "Puerto Rico", 78: "U.S. Virgin Islands"
}


# Dictionary 2: State Name -> FIPS Code (reverse mapping)
state_to_fips = {v: k for k, v in fips_to_state.items()}

# Optional: Dictionary 3: State Abbreviation -> FIPS Code
abbr_to_fips = {
    "AL": 1, "AK": 2, "AZ": 4, "AR": 5,
    "CA": 6, "CO": 8, "CT": 9, "DE": 10,
    "DC": 11, "FL": 12, "GA": 13, "HI": 15,
    "ID": 16, "IL": 17, "IN": 18, "IA": 19,
    "KS": 20, "KY": 21, "LA": 22, "ME": 23,
    "MD": 24, "MA": 25, "MI": 26, "MN": 27,
    "MS": 28, "MO": 29, "MT": 30, "NE": 31,
    "NV": 32, "NH": 33, "NJ": 34, "NM": 35,
    "NY": 36, "NC": 37, "ND": 38, "OH": 39,
    "OK": 40, "OR": 41, "PA": 42, "RI": 44,
    "SC": 45, "SD": 46, "TN": 47, "TX": 48,
    "UT": 49, "VT": 50, "VA": 51, "WA": 53,
    "WV": 54, "WI": 55, "WY": 56,
    "GU": 66, "PR": 72, "VI": 78
}

# Optional: Dictionary 4: FIPS Code -> State Abbreviation
fips_to_abbr = {v: k for k, v in abbr_to_fips.items()}

# Filter special_vars to only include columns that exist in the dataset
special_vars_existing = [v for v in special_vars if v in df.columns]
print(f"Special variables found in dataset: {len(special_vars_existing)} of {len(special_vars)}")

print(f"\nFIPS mapping loaded: {len(fips_to_state)} states/territories")
print(f"Special variables: {len(special_vars_existing)} columns with special missing code handling")

['PREDIAB2', 'DIFFDRES', 'CSRVRTRN', 'CELLFON5', '_HISPANC', '_INCOMG1', 'DIFFALON', 'SOMALE', '_RFSMOK3', '_RFBING6', 'CERVSCRN', 'HHADULT', 'INSULIN1', 'CHCSCNC1', '_DUALUSE', '_EDUCAG', '_AIDTST4', '_AGE65YR', 'LOADULK2', '_MICHD', '_BMI5CAT', 'MARIJAN1', '_AGE80', 'MARJVAPE', 'RENTHOM1', 'EXERANY2', 'CRVCLPAP', '_LLCPWT2', 'SDNATES1', 'SDHTRNSP', 'CCLGHOUS', '_ASTHMS1', '_STSTR', 'HADHYST2', 'USEMRJN4', 'PSATEST1', 'MARITAL', 'ACETOUCH', 'CELPHON1', 'STOPSMK2', 'MARJEAT', 'STOOLDN2', 'HEATTBCO', 'HIVTSTD3', 'CSRVCLIN', 'DIABTYPE', 'CAGEG', 'PREGNANT', 'COLGHOUS', '_SEX', 'PNEUVAC4', 'LCSFIRST', '_PNEUMO3', 'CHKHEMO3', '_METSTAT', 'HIVTST7', 'HTM4', 'CPDEMO1C', 'CHECKUP1', 'WTKG3', 'PERSDOC3', 'GENHLTH', '_FLSHOT7', 'PCPSARS2', '_AGE_G', 'INCOME3', 'CHILDREN', '_URBSTAT', 'HADMAM', 'CTELENM1', 'HADSIGM4', 'SDHEMPLY', 'CSRVDOC1', 'QSTVER', 'CSRVDEIN', '_RAWRAKE', 'SEXVAR', 'SAFETIME', 'MENTHLTH', 'GUNLOAD', 'PSATIME1', 'CSRVCTL2', '_CHISPNC', 'SHINGLE2', 'FEETSORE', 'DROCDY4_', 'ACED

In [6]:
def show_survey(idx):
    survey = df.iloc[idx, :]
    indices = survey.index
    
    for i in range(len(survey)):
        print(indices[i], ":", survey[i], "*" if survey[i] in standard_missing_codes else "")

show_survey(0)

PREDIAB2 : nan 
DIFFDRES : 2.0 
CSRVRTRN : nan 
CELLFON5 : nan 
_HISPANC : 2.0 
_INCOMG1 : 9.0 *
DIFFALON : 2.0 
SOMALE : nan 
_RFSMOK3 : 1.0 
_RFBING6 : 1.0 
CERVSCRN : 2.0 
HHADULT : nan 
INSULIN1 : nan 
CHCSCNC1 : 2.0 
_DUALUSE : 1.0 
_EDUCAG : 4.0 
_AIDTST4 : 2.0 
_AGE65YR : 2.0 
LOADULK2 : nan 
_MICHD : 2.0 
_BMI5CAT : nan 
MARIJAN1 : nan 
_AGE80 : 80.0 
MARJVAPE : nan 
RENTHOM1 : 1.0 
EXERANY2 : 2.0 
CRVCLPAP : nan 
_LLCPWT2 : 813.9185168510359 
SDNATES1 : nan 
SDHTRNSP : 2.0 
CCLGHOUS : nan 
_ASTHMS1 : 3.0 
_STSTR : 11011.0 
HADHYST2 : 2.0 
USEMRJN4 : nan 
PSATEST1 : nan 
MARITAL : 1.0 
ACETOUCH : nan 
CELPHON1 : 2.0 
STOPSMK2 : nan 
MARJEAT : nan 
STOOLDN2 : nan 
HEATTBCO : nan 
HIVTSTD3 : nan 
CSRVCLIN : nan 
DIABTYPE : nan 
CAGEG : nan 
PREGNANT : nan 
COLGHOUS : nan 
_SEX : 2.0 
PNEUVAC4 : 2.0 
LCSFIRST : nan 
_PNEUMO3 : 2.0 
CHKHEMO3 : nan 
_METSTAT : 1.0 
HIVTST7 : 2.0 
HTM4 : nan 
CPDEMO1C : 2.0 
CHECKUP1 : 1.0 
WTKG3 : nan 
PERSDOC3 : 1.0 
GENHLTH : 2.0 
_FLSHOT7 : 1.0 


C:\Users\Iman\AppData\Local\Temp\ipykernel_2980\1816932553.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(indices[i], ":", survey[i], "*" if survey[i] in standard_missing_codes else "")


In [7]:
df.head()

,PREDIAB2,DIFFDRES,CSRVRTRN,CELLFON5,_HISPANC,_INCOMG1,DIFFALON,SOMALE,_RFSMOK3,_RFBING6,...,MENTECIG,_BMI5,LCSCTSC1,STOLTEST,ASTHNOW,RCSGEND1,SOFEMALE,HOWLONG,CHCKDNY2,YEAR
0,NaN,2.0,NaN,NaN,2.0,9.0,2.0,NaN,1.0,1.0,...,NaN,NaN,2.0,NaN,NaN,NaN,NaN,1.0,2.0,2022
1,NaN,2.0,NaN,NaN,2.0,3.0,2.0,NaN,1.0,1.0,...,NaN,2657.0,2.0,NaN,NaN,NaN,NaN,4.0,2.0,2022
2,NaN,2.0,NaN,NaN,2.0,6.0,2.0,NaN,1.0,1.0,...,NaN,2561.0,2.0,NaN,NaN,NaN,NaN,1.0,2.0,2022
3,NaN,2.0,NaN,NaN,2.0,9.0,2.0,NaN,2.0,1.0,...,NaN,2330.0,1.0,NaN,1.0,NaN,NaN,2.0,2.0,2022
4,NaN,2.0,NaN,NaN,2.0,3.0,2.0,NaN,1.0,1.0,...,NaN,2177.0,1.0,NaN,NaN,NaN,NaN,1.0,2.0,2022


### Target

#### Typical Label

| Purpose | Choice |
|:---|:---|
| General health outcomes | **`GENHLTH`** |
| Subjective wellbeing, life quality, or happiness (4-point scale: Very Satisfied to Very Dissatisfied) | **`LSATISFY`** |
| Clinical prediction or chronic disease modeling | **`DIABETE4`** |
| Mental health frequency or severity (Count of days (0-30) mental health was not good in the past month) | **`MENTHLTH`** |

In [8]:
target = "MENTHLTH" # Number of mentally unhealthy days
target_ = "LSATISFY"

## Feature details

In [9]:
# ============================================================
# Dynamic Loading of SAS Label File
# Structure: data/2024/SASOUT24_LLCP.SAS, data/2023/SASOUT23_LLCP.SAS, ...
# ============================================================

# Find all SAS label files
sas_files = glob.glob("data/*/SASOUT??_LLCP.SAS")
print(f"Found {len(sas_files)} SAS label files:")
for f in sorted(sas_files):
    print(f"  {f}")

if not sas_files:
    # Fallback: try any .SAS file
    sas_files = glob.glob("data/*/*.SAS")
    print(f"Found {len(sas_files)} SAS files (fallback):")
    for f in sorted(sas_files):
        print(f"  {f}")

if not sas_files:
    raise FileNotFoundError("No SAS label file found in data/*/SASOUT??_LLCP.SAS")

# Use the most recent year (or any) for labels
# Typically variable labels are consistent across years
def extract_year_from_path(path):
    """Extract year from folder name or filename."""
    folder = os.path.basename(os.path.dirname(path))  # e.g., "2024"
    match = re.search(r'(\d{4})', folder)
    if match:
        return int(match.group(1))
    # Fallback: try filename
    filename = os.path.basename(path)
    match = re.search(r'(\d{4})', filename)
    if match:
        return int(match.group(1))
    return None

# Sort by year (descending) to get the latest
sas_files_sorted = sorted(sas_files, key=lambda x: extract_year_from_path(x) or 0, reverse=True)
chosen_sas = sas_files_sorted[0]
chosen_year = extract_year_from_path(chosen_sas)
print(f"\nUsing SAS file for labels: {chosen_sas} (year {chosen_year})")

# Read the SAS file
with open(chosen_sas, "r", encoding="latin1") as f:
    text = f.read()

# Extract LABEL section
label_pattern = r"([A-Z0-9_]+)\s*=\s*'([^']*)'"
matches = re.findall(label_pattern, text)

feature_details = {}

def pretty_label(label):
    return (
        label.replace("CALCULATED VARIABLE", "")
             .replace("COMPUTED", "")
             .strip()
             .title()
    )

feature_details = {
                    "YEAR": 
                        {"name": pretty_label("Dataset Year"), 
                         "desc": "Dataset Year".capitalize()},
                    "BMI":
                        {"name": pretty_label("BMI"), 
                         "desc": "Body Mass Index".capitalize()},
                  }

for var, label in matches:
    feature_details[var] = {
        "name": pretty_label(label),
        "desc": label.capitalize()
    }

print(f"Variables extracted: {len(feature_details)}")

Found 3 SAS label files:
  data\2022\SASOUT22_LLCP.SAS
  data\2023\SASOUT23_LLCP.SAS
  data\2024\SASOUT24_LLCP.SAS

Using SAS file for labels: data\2024\SASOUT24_LLCP.SAS (year 2024)
Variables extracted: 303


In [10]:
feature_details["GENHLTH"]

{'name': 'General Health', 'desc': 'General health'}

In [11]:
feature_details["HHADULT"]

{'name': 'Number Of Adults In Household',
 'desc': 'Number of adults in household'}

In [12]:
for feature in feature_details:
    desc = feature_details[feature]["desc"]
    if "satisf" in desc.lower():
        print(feature, desc)

LSATISFY Satisfaction with life


---

In [13]:
# ============================================================
# 1. Decode byte columns
# ============================================================
for c in df.columns:
    if df[c].dtype == object:
        try:
            df[c] = df[c].apply(lambda x: x.decode('utf-8') if isinstance(x, bytes) else x)
        except:
            pass

# ============================================================
# 2. Remove BRFSS Missing Codes
# ============================================================

# Apply standard codes to all columns
for col in df.columns:
    if col not in special_vars:
        df[col] = df[col].replace(standard_missing_codes, np.nan)

# Apply special handling to health days (convert 88 to 0)
for col in health_days_vars:
    if col in df.columns:
        df[col] = df[col].replace(special_missing_codes, np.nan)
        df[col] = df[col].replace(88, 0)  # 88 = 0 days
        # 7, 8, 9 remain as 7, 8, 9 days

# Apply special handling to other special variables
for col in special_vars:
    if col not in health_days_vars:
        if col in df.columns:
            df[col] = df[col].replace(special_missing_codes, np.nan)
            # 7, 8, 9, 88 remain as valid values

# ============================================================
# 3. Drop rows missing target
# ============================================================

df = df.dropna(subset=[target])
df = df.dropna(subset=[target_])

# ============================================================
# 4. BMI Scaling
# ============================================================
# The _BMI5 variable is stored as an integer (e.g., 2500, 3015).
# Divide by 100 to convert to standard BMI format (e.g., 25.0, 30.15).

if '_BMI5' in df.columns:
    df['BMI'] = df['_BMI5'] / 100.0
    
    print(f"BMI Conversion Complete. Example: _BMI5 = 2500 -> BMI = 25.0")
    print(f"BMI Column Summary:")
    print(df['BMI'].describe())

# ============================================================
# 5. Limit data: sample equally from each year
# ============================================================
if 'YEAR' in df.columns:
    # Recompute actual years present after cleaning
    actual_years = df['YEAR'].unique()
    n_years_actual = len(actual_years)
    print(f"Actual years in cleaned data: {n_years_actual}")
    
    # Count rows per year
    year_counts = df.groupby('YEAR').size()
    min_rows = year_counts.min()
    print(f"Rows per year:\n{year_counts}")
    print(f"Minimum rows among years: {min_rows}")
    
    # Desired samples per year (total n_samples divided by number of years)
    samples_per_year = int(n_samples / n_years_actual)
    print(f"Desired samples per year: {samples_per_year}")
    
    # Cap to minimum available to avoid sampling with replacement
    if samples_per_year > min_rows:
        print(f"Warning: samples_per_year ({samples_per_year}) exceeds min rows ({min_rows}). Using {min_rows}.")
        samples_per_year = min_rows
    
    # Sample from each year
    sampled_list = []
    for year in actual_years:
        df_year = df[df['YEAR'] == year]
        sampled = df_year.sample(n=samples_per_year, random_state=42)
        sampled_list.append(sampled)
    
    # Concatenate and shuffle
    df = pd.concat(sampled_list, ignore_index=True)
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    
    print(f"After sampling: {len(df)} rows (~{samples_per_year} per year, total {n_years_actual} years)")
    print(f"Rows per year after sampling:\n{df.groupby('YEAR').size()}")
else:
    print("No YEAR column; sampling total n_samples from entire dataset.")
    if len(df) > n_samples:
        df = df.sample(n=n_samples, random_state=42).reset_index(drop=True)

# ============================================================
# 6. Remove ID Columns
# ============================================================
drop_cols = [
    target,            # Target shouldn't be in the features
    target_,
    "SEQNO",           # Unique identifier
    "_PSU",            # Primary Sampling Unit (for survey design)
    "_STSTR",          # Stratification variable (for survey design)
    "IDATE",           # Interview date (may not be needed)
    "IMONTH",          # Interview month (may not be needed)
    "IDAY",            # Interview day (may not be needed)
    "IYEAR",            # Interview year (may not be needed)
    "FMONTH",
    "DISPCODE", 
    "LANDLINE",
]

X = df.drop(columns=drop_cols, errors="ignore")

# ============================================================
# 7. Drop redundant features according to the heatmap correlations
# ============================================================
exact_redundant_features = ["_WT2RAKE", "WTKG3", '_LLCPWT', '_LLCPWT2', '_STRWT', '_WT2RAKE', '_RAWRAKE', '_STSTR', 'WEIGHT2', 'WTKG3', 'HEIGHT3', 'HTIN4', 'HTM4']
redundant_features = []
for key in feature_details:
    if key[:4] == "_LLC" or (key[:4] == "_AGE" and key != "_AGE80") or key in exact_redundant_features:
        redundant_features.append(key)
print("Redundant features:", redundant_features)

X = X.drop(columns=redundant_features, errors="ignore")

# ============================================================
# 8. Drop no-sense features
# ============================================================
questionnaire_features = ["QSTLANG", "YEAR", "QSTVER", "CTELNUM1"]
print("Questionnaire features:", questionnaire_features)

X = X.drop(columns=questionnaire_features, errors="ignore")

# ============================================================
# 9. Correlated features
# ============================================================
correlated_features = {
    "_SMOKER3": ["_RFSMOK3",  "SMOKE100"],
    "_ASTHMS1": ["ASTHMA3", "_CASTHM1", "_LTASTH1"],
    "_URBSTAT": ["_METSTAT "],
    "PHYSHLTH": ["_PHYS14D"],
    "_STATE":   ["CSTATE1"],
    "CHILDREN": ["_CHLDCNT"],
    "EDUCA":    ["_EDUCAG"],
    "BMI":      ["_BMI5CAT", "_BMI5", "_RFBMI5"]
}
print("Correlated features:", correlated_features)

for correlated_feature in correlated_features:
    X = X.drop(columns=correlated_features[correlated_feature], errors="ignore")

# ============================================================
# 10. Remove leakage columns
#    These are CDC-derived variables based directly on the target
# ============================================================
if target == "MENTHLTH":
    leakage = ["_MENT14D", "POORHLTH"] # "PHYSHLTH" ?
elif target == "PHYSHLTH":
    leakage = ["_PHYS14D"]
elif target == "GENHLTH":
    leakage = ["_RFHLTH"]
elif target == "LSATISFY":
    leakage = []  # No known derived variable for LSATISFY
else:
    leakage = []

X = X.drop(columns=leakage, errors="ignore")

# Store y (target)
y = df[target]
y_ = df[target_]

# ============================================================
# 11. Remove Features with >50% Missing
# ============================================================
missing_pct = X.isnull().mean()
X = X.loc[:, missing_pct < 0.50]

print(f"Remaining Features: {len(X.columns)}")

# ============================================================
# 12. Convert Categoricals to Integers
# ============================================================
for col in X.columns:
    if X[col].dtype == object:
        X[col] = pd.factorize(X[col])[0]

# ============================================================
# 11. Impute Missing Values (Median Strategy)
# ============================================================
imputer = SimpleImputer(strategy="median")
X = pd.DataFrame(
    imputer.fit_transform(X),
    columns=X.columns
)

# ============================================================
# 13. (Optional) Handle Infinite Values
# ============================================================
X.replace([np.inf, -np.inf], np.nan, inplace=True)
X = pd.DataFrame(
    SimpleImputer(strategy="median").fit_transform(X),
    columns=X.columns
)

# ============================================================
# 14. (Optional) Convert Target if Needed
# ============================================================
# For binary classification targets (e.g., DIABETE4):
# if target == "DIABETE4":
#     # Convert to binary: 1 = Yes, 0 = No/Pre-diabetic/etc.
#     y = y.apply(lambda x: 1 if x == 1 else 0)

# For ordinal targets (e.g., GENHLTH):
# Keep as is (1-5 scale)

C:\Users\Iman\AppData\Local\Temp\ipykernel_2980\4095735683.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['BMI'] = df['_BMI5'] / 100.0


BMI Conversion Complete. Example: _BMI5 = 2500 -> BMI = 25.0
BMI Column Summary:
count    625099.000000
mean         28.592178
std           6.545163
min          12.020000
25%          24.210000
50%          27.440000
75%          31.750000
max          99.820000
Name: BMI, dtype: float64
Actual years in cleaned data: 3
Rows per year:
YEAR
2022    246845
2023    220510
2024    198750
dtype: int64
Minimum rows among years: 198750
Desired samples per year: 13333
After sampling: 39999 rows (~13333 per year, total 3 years)
Rows per year after sampling:
YEAR
2022    13333
2023    13333
2024    13333
dtype: int64
Redundant features: ['HEIGHT3', 'HTIN4', 'HTM4', 'WEIGHT2', 'WTKG3', '_AGE65YR', '_AGEG5YR', '_AGE_G', '_LLCPWT', '_LLCPWT2', '_RAWRAKE', '_STRWT', '_STSTR', '_WT2RAKE']
Questionnaire features: ['QSTLANG', 'YEAR', 'QSTVER', 'CTELNUM1']
Correlated features: {'_SMOKER3': ['_RFSMOK3', 'SMOKE100'], '_ASTHMS1': ['ASTHMA3', '_CASTHM1', '_LTASTH1'], '_URBSTAT': ['_METSTAT '], 'PHYSHLTH': 

In [14]:
common_cols

['PREDIAB2',
 'DIFFDRES',
 'CSRVRTRN',
 'CELLFON5',
 '_HISPANC',
 '_INCOMG1',
 'DIFFALON',
 'SOMALE',
 '_RFSMOK3',
 '_RFBING6',
 'CERVSCRN',
 'HHADULT',
 'INSULIN1',
 'CHCSCNC1',
 '_DUALUSE',
 '_EDUCAG',
 '_AIDTST4',
 '_AGE65YR',
 'LOADULK2',
 '_MICHD',
 '_BMI5CAT',
 'MARIJAN1',
 '_AGE80',
 'MARJVAPE',
 'RENTHOM1',
 'EXERANY2',
 'CRVCLPAP',
 '_LLCPWT2',
 'SDNATES1',
 'SDHTRNSP',
 'CCLGHOUS',
 '_ASTHMS1',
 '_STSTR',
 'HADHYST2',
 'USEMRJN4',
 'PSATEST1',
 'MARITAL',
 'ACETOUCH',
 'CELPHON1',
 'STOPSMK2',
 'MARJEAT',
 'STOOLDN2',
 'HEATTBCO',
 'HIVTSTD3',
 'CSRVCLIN',
 'DIABTYPE',
 'CAGEG',
 'PREGNANT',
 'COLGHOUS',
 '_SEX',
 'PNEUVAC4',
 'LCSFIRST',
 '_PNEUMO3',
 'CHKHEMO3',
 '_METSTAT',
 'HIVTST7',
 'HTM4',
 'CPDEMO1C',
 'CHECKUP1',
 'WTKG3',
 'PERSDOC3',
 'GENHLTH',
 '_FLSHOT7',
 'PCPSARS2',
 '_AGE_G',
 'INCOME3',
 'CHILDREN',
 '_URBSTAT',
 'HADMAM',
 'CTELENM1',
 'HADSIGM4',
 'SDHEMPLY',
 'CSRVDOC1',
 'QSTVER',
 'CSRVDEIN',
 '_RAWRAKE',
 'SEXVAR',
 'SAFETIME',
 'MENTHLTH',
 'GUNLOAD'

In [15]:
print(f"Final X shape: {X.shape}")
print(f"Final y shape: {y.shape}")
print(f"Target distribution:\n{y.value_counts().sort_index()}")

Final X shape: (39999, 68)
Final y shape: (39999,)
Target distribution:
MENTHLTH
0.0     24228
1.0      1352
2.0      2262
3.0      1460
4.0       780
5.0      1823
6.0       204
7.0       702
8.0       163
9.0        26
10.0     1390
11.0       16
12.0      122
13.0       17
14.0      251
15.0     1271
16.0       26
17.0       15
18.0       31
19.0        4
20.0      853
21.0       47
22.0       21
23.0       10
24.0       12
25.0      274
26.0        8
27.0       19
28.0       88
29.0       36
30.0     2488
Name: count, dtype: int64


In [31]:
dt = pd.concat([X, y, y_], axis=1)
dt.to_csv(f"data/preprocessed/{n_years}-years_{dt.shape[0]}-samples_{dt.shape[1]-2}-features_raw.csv", index=False)

---